In [1]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY2')

In [ ]:
import gradio as gr
import time
import os
import tempfile
import shutil
from google import genai
from google.genai import types
import threading
from datetime import datetime

# Initialize the client (you'll need to set your API key)
# GEMINI_API_KEY = "your_api_key_here"  # Replace with your actual API key
client = genai.Client(api_key=GEMINI_API_KEY)

def generate_video(prompt, filename, progress=gr.Progress()):
    """Generate video using Gemini API with progress tracking"""
    try:
        # Add timestamp to filename if not provided
        if not filename.strip():
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"generated_video_{timestamp}.mp4"
        elif not filename.endswith('.mp4'):
            filename += '.mp4'

        # Start video generation
        progress(0.1, desc="Starting video generation...")
        operation = client.models.generate_videos(
            model="veo-3.0-generate-preview",
            prompt=prompt,
        )

        # Poll the operation status until the video is ready
        progress(0.2, desc="Video generation in progress...")
        poll_count = 0
        while not operation.done:
            poll_count += 1
            progress(min(0.2 + (poll_count * 0.05), 0.9), desc=f"Generating video... (check {poll_count})")
            time.sleep(10)
            operation = client.operations.get(operation)

        # Download the generated video (this saves to local directory)
        progress(0.95, desc="Downloading generated video...")
        generated_video = operation.response.generated_videos[0]
        client.files.download(file=generated_video.video)
        generated_video.video.save(filename)

        # Now copy the file to a temp location that Gradio can serve
        if os.path.exists(filename):
            # Create a temporary file that Gradio can access
            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
            temp_path = temp_file.name
            temp_file.close()

            # Copy the generated video to temp location
            shutil.copy2(filename, temp_path)

            progress(1.0, desc="Video generation complete!")
            return temp_path, f"✅ Video generated successfully: {filename}"
        else:
            return None, f"❌ Error: Video file was not created"

    except Exception as e:
        return None, f"❌ Error generating video: {str(e)}"

def add_message_to_history(history, role, content, video_path=None):
    """Add a message to the chat history"""
    timestamp = datetime.now().strftime("%H:%M:%S")
    if video_path and os.path.exists(video_path):
        # Add video to the message
        history.append([f"[{timestamp}] {role}: {content}", None])
        history.append([None, (video_path,)])
    else:
        history.append([f"[{timestamp}] {role}: {content}", None])
    return history

def process_video_generation(prompt, filename, history):
    """Process video generation and update chat history"""
    if not prompt.strip():
        updated_history = add_message_to_history(history, "System", "Please enter a prompt for video generation.")
        return updated_history, "", ""

    # Add user message to history
    updated_history = add_message_to_history(history, "You", f"Generate video: {prompt}")

    # Add thinking message
    updated_history = add_message_to_history(updated_history, "AI", "🤔 Thinking... Video generation in progress. This may take several minutes...")

    return updated_history, "", ""

def generate_and_update(prompt, filename, history):
    """Generate video and update the interface"""
    try:
        # Generate the video
        video_path, status_message = generate_video(prompt, filename)

        # Remove the "thinking" message (last message)
        if history and "Thinking..." in str(history[-1]):
            history = history[:-1]

        # Add completion message
        if video_path and os.path.exists(video_path):
            updated_history = add_message_to_history(history, "AI", status_message, video_path)
        else:
            updated_history = add_message_to_history(history, "AI", status_message)

        return updated_history, video_path if video_path and os.path.exists(video_path) else None

    except Exception as e:
        # Remove the "thinking" message
        if history and "Thinking..." in str(history[-1]):
            history = history[:-1]

        error_message = f"❌ Error: {str(e)}"
        updated_history = add_message_to_history(history, "AI", error_message)
        return updated_history, None

# Create the Gradio interface
with gr.Blocks(title="AI Video Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🎬 AI Video Generator")
    gr.Markdown("Generate videos using AI with simple text prompts")

    with gr.Row():
        # Sidebar
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("## 📝 Video Generation Controls")

            with gr.Group():
                prompt_input = gr.Textbox(
                    label="Video Prompt",
                    placeholder="Describe the video you want to generate...",
                    lines=5,
                    info="Enter a detailed description of the video you want to create"
                )

                filename_input = gr.Textbox(
                    label="Output Filename",
                    placeholder="my_video.mp4",
                    info="Optional: Leave blank for auto-generated name"
                )

                generate_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg")

                gr.Markdown("### 💡 Tips:")
                gr.Markdown("""
                - Be specific about scenes, subjects, and actions
                - Include details about lighting, camera angles, and mood
                - Video generation may take 5-15 minutes
                - Check the chat for progress updates
                """)

        # Main chat area
        with gr.Column(scale=2):
            gr.Markdown("## 💬 Generation History")

            chatbot = gr.Chatbot(
                height=500,
                show_label=False,
                avatar_images=("👤", "🤖"),
                bubble_full_width=False
            )

            # Download section
            with gr.Row():
                download_file = gr.File(
                    label="📥 Download Generated Video",
                    visible=False
                )

    # Clear history button
    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear History", variant="secondary")

    # Event handlers
    def handle_generate(prompt, filename, history):
        # First update with thinking message
        updated_history, _, _ = process_video_generation(prompt, filename, history)
        yield updated_history, gr.update(visible=False)

        # Then generate video and update
        final_history, video_path = generate_and_update(prompt, filename, updated_history)
        if video_path:
            yield final_history, gr.update(value=video_path, visible=True)
        else:
            yield final_history, gr.update(visible=False)

    def clear_history():
        return [], gr.update(visible=False)

    # Connect events
    generate_btn.click(
        fn=handle_generate,
        inputs=[prompt_input, filename_input, chatbot],
        outputs=[chatbot, download_file],
        show_progress=True
    )

    clear_btn.click(
        fn=clear_history,
        outputs=[chatbot, download_file]
    )

    # Add enter key support for prompt input
    prompt_input.submit(
        fn=handle_generate,
        inputs=[prompt_input, filename_input, chatbot],
        outputs=[chatbot, download_file],
        show_progress=True
    )

if __name__ == "__main__":
    app.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        debug=True
    )

/tmp/ipython-input-3565260546.py:155: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-3565260546.py:155: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
